# Inspect a regional prediction

Same steps as `ps_density.pipeline.predict_region`: area of interest from `config/paths.yaml`,
the 2-channel paper checkpoint from Hugging Face (`dominika-malinowska/ps_predictions`)
unless `predict.checkpoint` is a local `.ckpt`, a local OSM PBF, `osmconvert`, and a public
coherence tile. This notebook also shows the infrastructure raster and the
coherence tile before prediction.

To rerun the pipeline with no extra figures:

```bash
poetry run python -m ps_density.pipeline.predict_region
```

Change the region in `predict.aoi_lon` and `predict.aoi_lat`. The region must sit
inside one 1x1 degree tile. Point `predict.osm_pbf` at a Geofabrik extract that
covers that region (the script does not download it).
Unlike the script, this notebook does not exit if a prediction GeoTIFF already exists.


In [ ]:
import math
import os
import time
from functools import partial

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import rasterio
import torch
from geocube.api.core import make_geocube
from geocube.rasterize import rasterize_image
from lightning.pytorch import seed_everything
from rasterio.plot import show
from shapely.geometry import box, mapping
from torch.utils.data import DataLoader
from torchgeo.datasets import stack_samples
from torchgeo.samplers import GridGeoSampler

from ps_density.utils.coordinates_selection import haversine_distance
from ps_density.config.paths import load_paths
from ps_density.utils.model_checkpoint import resolve_paper_checkpoint
from ps_density.utils.prediction_plots import (
    georreferenced_chip_generator,
    merge_georeferenced_chips,
)
from ps_density.utils.preprocessing_OSM_data import (
    BuildRoadRailHandler,
    convert_osm_osmconvert,
    map_value,
)
from ps_density.utils.torchgeo_classes_def import PixelwiseRegressionTask
from ps_density.utils.torchgeo_fun_to_read_input import RasterDataset_imgs, scale_coh

_cfg = load_paths()
x = list(_cfg.predict.aoi_lon)
y = list(_cfg.predict.aoi_lat)
print(
    "Area km2: {:.2f}".format(
        haversine_distance(y[0], x[0], y[1], x[0])
        * haversine_distance(y[0], x[0], y[0], x[1])
    )
)

experiment_dir = str(_cfg.predict.output.results_dir)
os.makedirs(experiment_dir, exist_ok=True)
osm_pbf = str(_cfg.predict.osm_pbf)
print("OSM PBF:", osm_pbf)

patch_size = 64
batch_size = 16
overlap = 8
seed = 54321
seed_everything(seed, workers=True)
generator = torch.Generator().manual_seed(seed)
model = PixelwiseRegressionTask.load_from_checkpoint(
    checkpoint_path=resolve_paper_checkpoint(_cfg.predict.checkpoint)
)

min_x, max_x = min(x), max(x)
min_y, max_y = min(y), max(y)
west, south = math.floor(min_x), math.floor(min_y)
lat_f = "N{:02d}".format(south + 1) if south >= 0 else "S{:02d}".format(abs(south + 1))
season = "summer" if south >= 0 else "winter"
lon_f = "E{:03d}".format(west) if west >= 0 else "W{:03d}".format(abs(west))
bounds = (min_x, min_y, max_x, max_y)
file_name = os.path.join(
    experiment_dir, "{}{}".format(lat_f, lon_f), "{}{}_PS_pred.tif".format(lat_f, lon_f)
)
infrmap_raster_dir = os.path.join(
    experiment_dir,
    "{}{}".format(lat_f, lon_f),
    "infr_map",
    "{}{}_infr_map.tif".format(lat_f, lon_f),
)
coh_tile_path = os.path.join(
    experiment_dir,
    "{}{}".format(lat_f, lon_f),
    "coh",
    "{lat}{lon}_{season}_vv_rho.tif".format(lat=lat_f, lon=lon_f, season=season),
)
print(lat_f, lon_f, season)
print("prediction:", file_name)


## Infrastructure map

Builds the raster with `BuildRoadRailHandler`, `convert_osm_osmconvert`, and `map_value`
when it does not already exist.


In [ ]:
if os.path.exists(infrmap_raster_dir):
    print("Infrastructure map already exists:", infrmap_raster_dir)
else:
    clipped_o5m = os.path.join(
        experiment_dir,
        "{}{}".format(lat_f, lon_f),
        "clipped_{}_{}.o5m".format(lat_f, lon_f),
    )
    if not os.path.exists(clipped_o5m):
        convert_osm_osmconvert(osm_pbf, clipped_o5m)
    handler = BuildRoadRailHandler()
    handler.apply_file(clipped_o5m, locations=True, idx="flex_mem")
    df_buil = pd.DataFrame(handler.buildings)
    df_road = pd.DataFrame(handler.roads)
    df_rail = pd.DataFrame(handler.railways)
    df_rail["type"] = "rail"
    df_buil["type"] = "building"
    df_road.rename(columns={"highway": "type"}, inplace=True)
    df_all = pd.concat([df_rail, df_road, df_buil])
    df_all["value"] = df_all.apply(map_value, axis=1)
    df_all.sort_values(by=["value"], inplace=True)
    gdf_OSM = gpd.GeoDataFrame(df_all)
    cube = make_geocube(
        gdf_OSM,
        measurements=["value"],
        resolution=(-1 / 1200, 1 / 1200),
        geom=mapping(box(min_x, min_y, max_x, max_y)),
        fill=0,
        rasterize_function=partial(rasterize_image, all_touched=True),
    )
    os.makedirs(os.path.dirname(infrmap_raster_dir), exist_ok=True)
    cube.value.rio.to_raster(infrmap_raster_dir)
    print("Wrote", infrmap_raster_dir)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
with rasterio.open(infrmap_raster_dir) as src:
    show(src, ax=ax, cmap="viridis", title="Infrastructure map")



## Long-term coherence

Downloads the public Sentinel-1 global coherence tile when it is not already on disk.
Requires the AWS CLI (`aws s3 cp --no-sign-request`).


In [ ]:
if os.path.exists(coh_tile_path):
    print("Coherence tile already exists:", coh_tile_path)
else:
    os.makedirs(os.path.dirname(coh_tile_path), exist_ok=True)
    command = (
        "aws s3 cp s3://sentinel-1-global-coherence-earthbigdata/data/tiles/"
        "{lat}{lon}/{lat}{lon}_{season}_vv_rho.tif --no-sign-request {dest}"
    ).format(lat=lat_f, lon=lon_f, season=season, dest=coh_tile_path)
    print(command)
    os.system(command)
    print("exists:", os.path.exists(coh_tile_path))


In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
with rasterio.open(coh_tile_path) as src:
    show(src, ax=ax, cmap="turbo", title="Long-term coherence")


## Predict

Skips the GPU prediction when the GeoTIFF is already present, and still leaves the
path printed so you can open it. Delete that file if you want to predict again.


In [ ]:
if os.path.exists(file_name):
    print("Prediction already exists:", file_name)
else:
    os.makedirs(os.path.dirname(file_name), exist_ok=True)
    imgs_input = RasterDataset_imgs(
        paths=os.path.dirname(coh_tile_path), transforms=scale_coh
    ) & RasterDataset_imgs(paths=os.path.dirname(infrmap_raster_dir))
    dataloader = DataLoader(
        dataset=imgs_input,
        batch_size=batch_size,
        sampler=GridGeoSampler(imgs_input, patch_size, patch_size - overlap),
        num_workers=0,
        collate_fn=stack_samples,
        generator=generator,
    )
    model.eval()
    start = time.time()
    chips = georreferenced_chip_generator(
        dataloader, model, imgs_input.crs.to_epsg(), imgs_input.res
    )
    print("predict seconds", time.time() - start)
    merge_georeferenced_chips(chips, file_name, overlap, bounds)
    print("wrote", file_name)
